# 규제 실습

**Regularization · Ridge · Lasso · 정규화**

계수의 크기를 제한하는 항을 더해 과적합을 억제하는 방법.

소재 분야에서 이해하기: 기술자가 많을 때 Lasso로 중요한 항만 남긴다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 선형 모델 문서](https://scikit-learn.org/stable/modules/linear_model.html)

## 1. 변수가 많을 때

무관한 변수를 많이 섞어놓고 규제의 효과를 봅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

n, useful, junk = 120, 5, 60
X_all = rng.normal(0, 1, (n, useful + junk))
coefficients = np.zeros(useful + junk)
coefficients[:useful] = [3.0, -2.0, 1.5, 0.0, 2.5]
y_all = X_all @ coefficients + rng.normal(0, 1.0, n)
print('시료 %d개, 변수 %d개 (실제로 의미 있는 변수는 %d개)' % (n, useful + junk, (coefficients != 0).sum()))

In [ ]:
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.model_selection import cross_val_score

for name, model in [('규제 없음', LinearRegression()),
                    ('Ridge(L2)', RidgeCV(alphas=np.logspace(-3, 3, 30))),
                    ('Lasso(L1)', LassoCV(cv=5, random_state=0, max_iter=20000))]:
    score = cross_val_score(model, X_all, y_all, cv=5, scoring='r2').mean()
    fitted = model.fit(X_all, y_all)
    nonzero = int(np.sum(np.abs(fitted.coef_) > 1e-6))
    print('%-11s 교차검증 R2 %6.3f  0이 아닌 계수 %d개' % (name, score, nonzero))

In [ ]:
lasso = LassoCV(cv=5, random_state=0, max_iter=20000).fit(X_all, y_all)
plt.stem(coefficients, linefmt='k-', markerfmt='ko', basefmt=' ', label='true')
plt.stem(lasso.coef_, linefmt='r-', markerfmt='rx', basefmt=' ', label='Lasso')
plt.xlabel('feature index'); plt.ylabel('coefficient'); plt.legend(); plt.show()

## 2. 해석

규제가 없으면 무관한 변수에도 계수가 붙어 새 데이터에서 무너집니다. L2는 계수를 전체적으로
줄이고, L1은 상당수를 정확히 0으로 만들어 변수 선택 효과를 냅니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#regularization)을 여세요.